### Week 36

In [ ]:
import pandas as pd

In [ ]:
from datasets import load_dataset
dataset = load_dataset("coastalcph/tydi_xor_rc")
train_set = dataset["train"].to_pandas()
validation_set = dataset["validation"].to_pandas()
languages = ["te", "ar", "ko"]
train_set = train_set[train_set["lang"].isin(languages)]
validation_set = validation_set[validation_set["lang"].isin(languages)]

In [ ]:
validation_set.shape

(1155, 7)

In [ ]:
train_set.shape

(6335, 7)

In [ ]:
train_set.columns


Index(['question', 'context', 'lang', 'answerable', 'answer_start', 'answer',
       'answer_inlang'],
      dtype='object')

In [ ]:
strip_punctuation = lambda line: [word.strip(string.punctuation+"؟")for word in line.split(" ")]
count_words = lambda line: len(line)

In [ ]:
import torch
print("Torch:", torch.__version__)        # should show +cu126
print("CUDA runtime:", torch.version.cuda) # '12.6'
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.8.0+cu126
CUDA runtime: 12.6
CUDA available: False


In [ ]:
%pip install deep_translator -q

In [ ]:
from deep_translator import GoogleTranslator

def translate_to_english(text):
    """Translates text to English using Google Translator."""
    try:
        translated_text = GoogleTranslator(source='auto', target='en').translate(text)
        return translated_text if translated_text else text # Return original if translation is empty
    except Exception as e:
        print(f"Translation failed for text: {text[:50]}... Error: {e}", text.shape)
        return text # Return original text in case of failure


In [ ]:
def translate(df,lang = "en"):
  for index, row in df.iterrows():
    print(row)
    if row['lang'] != lang:
        df.loc[index, 'question_translated'] = translate_to_english(row['question'])
        # df.loc[index, 'answer_inlang'] = translate_to_english(row['answer_inlang'])
        df.loc[index, 'translated'] = True
    else:
        df.loc[index, 'translated'] = False

In [ ]:
# train_tr = translate(train_set)
# val_tr = translate(validation_set)

## HERE LOAD THE FILES

In [5]:
val_tr = pd.read_csv("data/validation_translated.csv")
train_tr = pd.read_csv("data/train_translated.csv")
test_df = pd.read_json("data/test.json")

### Week 36 - rule-based classifier


In [1]:
%pip install deep_translator -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.3 MB/s eta 0:00:00


In [6]:
import pandas as pd
import string
import numpy as np
from deep_translator import GoogleTranslator

In [7]:
train_tr.head()

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated
0,4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,NaN,"['30년', '전쟁의', '승자는', '누구인가']",4,"['The', 'conflict', 'between', 'France', 'and'...",108,Who is the winner of the Thirty Years' War?
1,4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,NaN,"['엑스선은', '누가', '발견하였는가']",3,"['X-rays', 'make', 'up', 'X-radiation', 'a', '...",122,Who discovered X-rays?
2,4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,NaN,"['아테네에서', '언제', '가장', '최근의', '올림픽이', '올렸나요']",6,"['In', '2022', 'Beijing', 'will', 'become', 't...",197,When was the last Olympic Games held in Athens?
3,4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),NaN,"['세상에서', '가장', '오래된', '방송사는', '무엇인가']",5,"['The', 'British', 'Broadcasting', 'Corporatio...",70,What's the oldest broadcaster in the world?
4,4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,NaN,"['팔레스타인', '수도는', '어딘가요']",3,"['Palestine', '', '', 'officially', 'the', 'St...",84,Where's the Palestinian capital?


In [8]:
val_tr.head()

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated
0,0,ఒరెగాన్ రాష్ట్రంలోని అతిపెద్ద నగరం ఏది ?,Portland is the largest city in the U.S. state...,te,True,0,Portland,NaN,"['ఒరెగాన్', 'రాష్ట్రంలోని', 'అతిపెద్ద', 'నగరం'...",6,"['Portland', 'is', 'the', 'largest', 'city', '...",117,What is the largest city in the state of Oregon?
1,1,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,"The word cholera is from ""kholera"" from χολή ""...",te,True,99,Indian subcontinent,NaN,"['కలరా', 'వ్యాధిని', 'మొదటగా', 'ఏ', 'దేశంలో', ...",7,"['The', 'word', 'cholera', 'is', 'from', 'khol...",117,In which country was cholera first discovered?
2,2,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,Since it became widespread in the 19th century...,te,True,451,England,NaN,"['కలరా', 'వ్యాధిని', 'మొదటగా', 'ఏ', 'దేశంలో', ...",7,"['Since', 'it', 'became', 'widespread', 'in', ...",122,In which country was cholera first discovered?
3,3,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,World War I occurred from 1914 to 1918. In ter...,te,True,26,1914,NaN,"['మొదటి', 'ప్రపంచ', 'యుద్ధం', 'ఎప్పుడు', 'మొదల...",6,"['World', 'War', 'I', 'occurred', 'from', '191...",188,When did World War I begin?
4,4,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,"World War I (often abbreviated as WWI or WW1),...",te,True,155,28 July 1914,NaN,"['మొదటి', 'ప్రపంచ', 'యుద్ధం', 'ఎప్పుడు', 'మొదల...",6,"['World', 'War', 'I', 'often', 'abbreviated', ...",115,When did World War I begin?


In [17]:
test_df

,question,context,lang,answerable,answer_start,answer,answer_inlang,translated,question_translated
0,When was the Kyivan Rus' founded?,Kyivan Rus' was a federation of Slavic tribes ...,en,True,64,late 9th century,late 9th century,False,NaN
1,Which city was the political center of Kyivan ...,Kyivan Rus' was a federation of Slavic tribes ...,en,True,89,Kyiv,Kyiv,False,NaN
2,Which prince converted Kyivan Rus' to Christia...,"In 988, Prince Volodymyr the Great adopted Chr...",en,True,9,Prince Volodymyr the Great,Prince Volodymyr the Great,False,NaN
3,When did Ukraine declare independence from the...,Ukraine declared its independence from the Sov...,en,True,60,24 August 1991,24 August 1991,False,NaN
4,Which empire ruled most of western Ukraine bef...,"Before World War I, western Ukraine was part o...",en,True,56,Austro-Hungarian Empire,Austro-Hungarian Empire,False,NaN
5,Was the Holodomor a man-made famine in Soviet ...,The Holodomor was a man-made famine that took ...,en,True,4,Yes,Yes,False,NaN
6,Did Ukraine become independent in 1989?,Ukraine declared its independence from the Sov...,en,True,60,No,No,False,NaN
7,Who was the first President of independent Ukr...,Ukraine declared its independence from the Sov...,en,False,-1,no answer,None,False,NaN
8,Did the Cossack Hetmanate sign a treaty with t...,The Pereyaslav Agreement of 1654 was a treaty ...,en,True,44,Yes,Yes,False,NaN
9,Who was the leader of the Ukrainian Insurgent ...,"During World War II, Ukraine became a major ba...",en,False,-1,no answer,None,False,NaN


In [9]:
strip_first = lambda x: x.split()[0]

In [10]:
import re

question_contractions = {
    "what's": "what is",
    "where's": "where is",
    "who's": "who is",
    "when's": "when is",
    "why's": "why is",
    "how's": "how is",
    "in what year": "when",
    "in what month": "when",
    "in what day": "when",
    "in what hour": "when",
    "in what minute": "when",
    "in what second": "when",
    "what're": "what are",
    "where're": "where are",
    "who're": "who are",
    "when're": "when are",
    "why're": "why are",
    "how're": "how are",
    "what've": "what have",
    "where've": "where have",
    "who've": "who have",
    "when've": "when have",
    "why've": "why have",
    "how've": "how have",
    "in which country": "where",
    "in which city": "where",
    "in which state": "where",
    "in which region": "where",
    "in which department": "where",
    "in what country": "where",
    "in what city": "where",
    "in what state": "where",
    "in what region": "where",
    "in what department": "where",
    "in what company": "where",
    "in what years": "when",
    "in what months": "when",
    "in what days": "when",
    "in what hours": "when",
    "in what minutes": "when",
    "in what seconds": "when",
    "in which year": "when",
    "in which month": "when",
    "in which day": "when",
    "in which hour": "when",
    "in which minute": "when",
    "in which second": "when",
    "in which direction": "where",
    "on what day": "when",
    "on what date": "when",
    "on what time": "when",

}

def expand_contractions(text, contractions=question_contractions):
    pattern = re.compile(r'\b(' + '|'.join(re.escape(k) for k in contractions.keys()) + r')\b', flags=re.IGNORECASE)
    return pattern.sub(lambda x: contractions[x.group().lower()], text)


In [11]:
train_tr["question_start"] = train_tr["question_translated"].apply(expand_contractions).apply(strip_first).apply(str.lower)
val_tr["question_start"] = val_tr["question_translated"].apply(expand_contractions).apply(strip_first).apply(str.lower)


In [12]:
def translate_to_english(text):
    """Translates text to English using Google Translator."""
    try:
        translated_text = GoogleTranslator(source='auto', target='en').translate(text)
        return translated_text if translated_text else text # Return original if translation is empty
    except Exception as e:
        print(f"Translation failed for text: {text[:50]}... Error: {e}", text.shape)
        return text # Return original text in case of failure


In [22]:
def translate(df, lang="en"):
    # Create boolean mask for rows that need translation
    needs_translation = df['lang'] != lang

    # Translate only the rows that need it
    df.loc[needs_translation, 'question_translated'] = df.loc[needs_translation, 'question'].apply(translate_to_english)
    df.loc[~needs_translation, 'question_translated'] = df.loc[~needs_translation, 'question']

    # Set translated flag
    df['translated'] = needs_translation

    return df

In [23]:
test_tr = test_df.copy()        # ← make a real copy
test_tr = translate(test_tr)    # ← now this returns df
test_tr

,question,context,lang,answerable,answer_start,answer,answer_inlang,translated,question_translated
0,When was the Kyivan Rus' founded?,Kyivan Rus' was a federation of Slavic tribes ...,en,True,64,late 9th century,late 9th century,False,When was the Kyivan Rus' founded?
1,Which city was the political center of Kyivan ...,Kyivan Rus' was a federation of Slavic tribes ...,en,True,89,Kyiv,Kyiv,False,Which city was the political center of Kyivan ...
2,Which prince converted Kyivan Rus' to Christia...,"In 988, Prince Volodymyr the Great adopted Chr...",en,True,9,Prince Volodymyr the Great,Prince Volodymyr the Great,False,Which prince converted Kyivan Rus' to Christia...
3,When did Ukraine declare independence from the...,Ukraine declared its independence from the Sov...,en,True,60,24 August 1991,24 August 1991,False,When did Ukraine declare independence from the...
4,Which empire ruled most of western Ukraine bef...,"Before World War I, western Ukraine was part o...",en,True,56,Austro-Hungarian Empire,Austro-Hungarian Empire,False,Which empire ruled most of western Ukraine bef...
5,Was the Holodomor a man-made famine in Soviet ...,The Holodomor was a man-made famine that took ...,en,True,4,Yes,Yes,False,Was the Holodomor a man-made famine in Soviet ...
6,Did Ukraine become independent in 1989?,Ukraine declared its independence from the Sov...,en,True,60,No,No,False,Did Ukraine become independent in 1989?
7,Who was the first President of independent Ukr...,Ukraine declared its independence from the Sov...,en,False,-1,no answer,None,False,Who was the first President of independent Ukr...
8,Did the Cossack Hetmanate sign a treaty with t...,The Pereyaslav Agreement of 1654 was a treaty ...,en,True,44,Yes,Yes,False,Did the Cossack Hetmanate sign a treaty with t...
9,Who was the leader of the Ukrainian Insurgent ...,"During World War II, Ukraine became a major ba...",en,False,-1,no answer,None,False,Who was the leader of the Ukrainian Insurgent ...


In [25]:

test_tr["question_start"] = (
    test_tr["question_translated"]
    .apply(expand_contractions)
    .apply(strip_first)
    .str.lower()
)

In [26]:
test_tr

,question,context,lang,answerable,answer_start,answer,answer_inlang,translated,question_translated,question_start
0,When was the Kyivan Rus' founded?,Kyivan Rus' was a federation of Slavic tribes ...,en,True,64,late 9th century,late 9th century,False,When was the Kyivan Rus' founded?,when
1,Which city was the political center of Kyivan ...,Kyivan Rus' was a federation of Slavic tribes ...,en,True,89,Kyiv,Kyiv,False,Which city was the political center of Kyivan ...,which
2,Which prince converted Kyivan Rus' to Christia...,"In 988, Prince Volodymyr the Great adopted Chr...",en,True,9,Prince Volodymyr the Great,Prince Volodymyr the Great,False,Which prince converted Kyivan Rus' to Christia...,which
3,When did Ukraine declare independence from the...,Ukraine declared its independence from the Sov...,en,True,60,24 August 1991,24 August 1991,False,When did Ukraine declare independence from the...,when
4,Which empire ruled most of western Ukraine bef...,"Before World War I, western Ukraine was part o...",en,True,56,Austro-Hungarian Empire,Austro-Hungarian Empire,False,Which empire ruled most of western Ukraine bef...,which
5,Was the Holodomor a man-made famine in Soviet ...,The Holodomor was a man-made famine that took ...,en,True,4,Yes,Yes,False,Was the Holodomor a man-made famine in Soviet ...,was
6,Did Ukraine become independent in 1989?,Ukraine declared its independence from the Sov...,en,True,60,No,No,False,Did Ukraine become independent in 1989?,did
7,Who was the first President of independent Ukr...,Ukraine declared its independence from the Sov...,en,False,-1,no answer,None,False,Who was the first President of independent Ukr...,who
8,Did the Cossack Hetmanate sign a treaty with t...,The Pereyaslav Agreement of 1654 was a treaty ...,en,True,44,Yes,Yes,False,Did the Cossack Hetmanate sign a treaty with t...,did
9,Who was the leader of the Ukrainian Insurgent ...,"During World War II, Ukraine became a major ba...",en,False,-1,no answer,None,False,Who was the leader of the Ukrainian Insurgent ...,who


#### Rule-based

In [27]:
def top_10(df):
    counts = (
        df.groupby(["answerable", "question_start"])
        .size()
        .rename("count")
        .reset_index()
    )

    counts["answerable"] = counts["answerable"].astype(bool)

    top10 = (
        counts.sort_values(["answerable", "count"], ascending=[True, False])\
              .groupby("answerable", group_keys=False)\
              .apply(lambda x: x) \
              .reset_index(drop=True) # Convert back to DataFrame
    )

    opposite_counts = counts.copy()
    opposite_counts["answerable"] = ~opposite_counts["answerable"]
    opposite_counts = opposite_counts.rename(columns={"count": "count_in_opposite_class"})

    top10_with_opposite = (
        top10.merge(
            opposite_counts[["answerable", "question_start", "count_in_opposite_class"]],
            on=["answerable", "question_start"],
            how="left",
        )
        .fillna({"count_in_opposite_class": 0})
    )
    top10_with_opposite["count_in_opposite_class"] = (
        top10_with_opposite["count_in_opposite_class"].astype(int)
    )

    top10_with_opposite["valid"] = np.select(
        [
            top10_with_opposite["count"] > top10_with_opposite["count_in_opposite_class"],
            top10_with_opposite["count"] < top10_with_opposite["count_in_opposite_class"],
        ],
        [True, False],
        default="tie",
    )
    return top10_with_opposite
# top10_with_opposite["is_more_in_answerable"] = np.where(
#     top10_with_opposite["count"] == top10_with_opposite["count_in_opposite_class"],
#     np.nan,
#     top10_with_opposite["count"] > top10_with_opposite["count_in_opposite_class"],
# )



In [28]:
valid = top_10(val_tr)
train = top_10(train_tr)

/tmp/ipython-input-2404154244.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x) \
/tmp/ipython-input-2404154244.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x) \


In [29]:
# train[train["answerable"]==False].head(6).to_latex()

In [30]:
# train[train["answerable"]==True].head().to_latex()

In [31]:
train_start_nonans = train[(train["valid"] == "True") & (train["answerable"] == False)]["question_start"].tolist()

In [32]:
train_start_nonans

['is',
 'can',
 'does',
 'are',
 'do',
 'has',
 'did',
 'was',
 'could',
 'at',
 'have',
 'were',
 'jack',
 'major,']

In [33]:
def rule_based(df, non_questions):
  df["prediction_answarable"] = np.where(df["question_start"].isin(non_questions), False, True)
  return df

In [34]:
train_pred = rule_based(train_tr, train_start_nonans)
val_pred = rule_based(val_tr, train_start_nonans)

In [35]:
train_pred["accurate"] = train_pred["answerable"] == train_pred["prediction_answarable"]
val_pred["accurate"] = val_pred["answerable"] == val_pred["prediction_answarable"]

In [36]:
test_pred = rule_based(test_tr, train_start_nonans)
test_pred["accurate"] = test_pred["answerable"] == test_pred["prediction_answarable"]

In [37]:
def metrics_with_bacc(y_true, y_pred, name="set"):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    # Confusion matrix components (binary, positive class = 1)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    # Safe div helper
    div = lambda a, b: (a / b) if b else 0.0

    accuracy  = div(tp + tn, tp + tn + fp + fn)
    precision = div(tp, tp + fp)
    recall    = div(tp, tp + fn)
    f1        = div(2 * precision * recall, precision + recall)

    tpr_pos = recall
    tpr_neg = div(tn, tn + fp)
    balanced_accuracy = 0.5 * (tpr_pos + tpr_neg)

    print(f"{name} accuracy: {accuracy:.4f},f1: {f1:.4f},baac: {balanced_accuracy:.4f}")
    # print(f"{name} precision: {precision:.4f}")
    # print(f"{name} recall: {recall:.4f}")

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "balanced_accuracy": balanced_accuracy,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }

def accuracy_and_f1(series, name = "sets"):
    accuracy = series.mean()
    tp = series.sum()
    fp = (series == False).sum()
    fn = (series == True).sum() - tp
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * precision * recall / (precision + recall)
    print(f"{name} accuracy: {accuracy:.4f}, f1: {f1:.4f}")
    #print(f"{name} precision: {precision:.4f}")
    #print(f"{name} recall: {recall:.4f}")


In [38]:
LANG = {"ar": "Arabic", "ko": "Korean", "te": "Telugu"}

In [42]:
for l in LANG.keys():
    metrics_with_bacc(train_pred[train_pred["lang"]==l]["answerable"],train_pred[train_pred["lang"]==l]["prediction_answarable"], f"Train data {LANG.get(l)}")
    metrics_with_bacc(val_pred[val_pred["lang"]==l]["answerable"],val_pred[val_pred["lang"]==l]["prediction_answarable"], f"Validation data {LANG.get(l)}")

metrics_with_bacc(train_pred["answerable"],train_pred["prediction_answarable"], "Train data")
metrics_with_bacc(val_pred["answerable"],val_pred["prediction_answarable"], "Validation data")
metrics_with_bacc(test_pred["answerable"],test_pred["prediction_answarable"], "Test data")
metrics_with_bacc(test_pred[test_pred["lang"]=="te"]["answerable"],test_pred[test_pred["lang"]==l]["prediction_answarable"], f"Test data {LANG.get(l)}")

Train data Arabic accuracy: 0.9644,f1: 0.9799,baac: 0.9646
Validation data Arabic accuracy: 0.9807,f1: 0.9889,baac: 0.9725
Train data Korean accuracy: 0.9756,f1: 0.9874,baac: 0.9180
Validation data Korean accuracy: 0.9635,f1: 0.9805,baac: 0.9062
Train data Telugu accuracy: 0.9675,f1: 0.9835,baac: 0.5218
Validation data Telugu accuracy: 0.7500,f1: 0.8567,baac: 0.4985
Train data accuracy: 0.9694,f1: 0.9837,baac: 0.9048
Validation data accuracy: 0.8987,f1: 0.9432,baac: 0.6942
Test data accuracy: 0.5938,f1: 0.7111,baac: 0.5136
Test data Telugu accuracy: 0.6000,f1: 0.6667,baac: 0.5833


{'accuracy': 0.6,
 'precision': 0.6666666666666666,
 'recall': 0.6666666666666666,
 'f1': 0.6666666666666666,
 'balanced_accuracy': 0.5833333333333333,
 'tp': 4,
 'tn': 2,
 'fp': 2,
 'fn': 2}

In [40]:
valid[(valid["valid"] == "True") & (valid["answerable"] == False)]

,answerable,question_start,count,count_in_opposite_class,valid
1,False,is,29,12,True
4,False,can,9,0,True
5,False,does,9,1,True
6,False,are,7,1,True
9,False,"bharat,",5,0,True
10,False,do,5,0,True
13,False,as,3,2,True
14,False,has,3,0,True
16,False,was,2,0,True


In [41]:
valid[valid["valid"]=="True"]

,answerable,question_start,count,count_in_opposite_class,valid
1,False,is,29,12,True
4,False,can,9,0,True
5,False,does,9,1,True
6,False,are,7,1,True
9,False,"bharat,",5,0,True
10,False,do,5,0,True
13,False,as,3,2,True
14,False,has,3,0,True
16,False,was,2,0,True
19,True,what,350,35,True
